# Kim et al. (2015) - Bio-Inspired Edge Detection

**Study**: Kim et al., 2015  
**Bio-Inspired Features**: LGN + V1  
**Architecture**: Center-surround receptive fields with simple/complex cells

This implements LGN center-surround mechanisms and V1 orientation-selective filters.

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python', 'numpy', 'matplotlib', 'tqdm', 'scikit-learn'], check=False)
import cv2, numpy as np, json
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR = Path('outputs') / 'Kim_2015'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = Path('..') / 'datasets' / 'HED_Small'

## LGN Center-Surround + V1 Orientation Filters

In [ ]:
def dog_filter(img, sigma1=1.0, sigma2=2.0):
    """LGN: Difference of Gaussians (center-surround)"""
    g1 = cv2.GaussianBlur(img, (0,0), sigma1)
    g2 = cv2.GaussianBlur(img, (0,0), sigma2)
    return g1 - g2

def gabor_filters_v1(img, orientations=8):
    """V1: Orientation-selective filters"""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img
    responses = []
    for theta in np.linspace(0, np.pi, orientations, endpoint=False):
        kernel = cv2.getGaborKernel((21, 21), 4.0, theta, 10.0, 0.5, 0, ktype=cv2.CV_32F)
        filtered = cv2.filter2D(gray, cv2.CV_32F, kernel)
        responses.append(np.abs(filtered))
    return np.max(responses, axis=0)

def kim_2015_edge_detector(img):
    """Kim et al. 2015: LGN + V1 processing"""
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img
    
    # LGN processing
    dog = dog_filter(gray)
    
    # V1 processing
    v1_response = gabor_filters_v1(img)
    
    # Combine
    combined = 0.5 * np.abs(dog) + 0.5 * v1_response
    return cv2.normalize(combined, None, 0, 1, cv2.NORM_MINMAX)

In [ ]:
# Load test images
img_dir = DATASET_ROOT / 'test' / 'images'
gt_dir = DATASET_ROOT / 'test' / 'edges'
images = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))[:20]

predictions, ground_truths = [], []
for img_path in tqdm(images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    gt_path = gt_dir / img_path.name.replace('.jpg', '.png')
    gt = cv2.imread(str(gt_path), 0).astype(np.float32) / 255.0 if gt_path.exists() else np.zeros(img.shape[:2], dtype=np.float32)
    
    pred = kim_2015_edge_detector(img)
    predictions.append(pred)
    ground_truths.append(gt)

print(f"✓ Processed {len(predictions)} images")

In [ ]:
# Compute metrics
def compute_metrics(preds, labels):
    t, ois, all_p, all_l = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l_bin = cv2.dilate((l>0.5).astype(np.float32), np.ones((3,3))).flatten()
        p_smooth = cv2.GaussianBlur(p, (3,3), 0).flatten()
        all_p.append(p_smooth); all_l.append(l_bin)
        ois.append(max([2*np.sum((p_smooth>=th)*l_bin)/(2*np.sum((p_smooth>=th)*l_bin)+np.sum((p_smooth>=th)*(1-l_bin))+np.sum((p_smooth<th)*l_bin)+1e-8) for th in t]))
    fp, fl = np.concatenate(all_p), np.concatenate(all_l)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': float(ods[0]), 'ODS_thresh': float(ods[1]), 'OIS': float(np.mean(ois)), 'AP': float(average_precision_score(fl, fp))}

m = compute_metrics(predictions, ground_truths)
print(f"\nKim et al. 2015: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f}")

with open(OUTPUT_DIR / 'kim_2015_metrics.json', 'w') as f:
    json.dump({'model': 'Kim et al. 2015', 'bio': 'LGN + V1', 'features': 'Center-surround + Orientation', 'metrics': m}, f, indent=2)
print("✅ Complete!")